# Demos: Lecture 18

In [ ]:
import pennylane as qml
from pennylane import numpy as np
import matplotlib.pyplot as plt

## Demo 1: depolarizing noise

PennyLane uses the following definition of depolarizing channel:

\begin{equation*}
\Phi(\rho) = (1-p) \rho + \frac{p}{3} X\rho X + \frac{p}{3} Y \rho Y +
\frac{p}{3} Z \rho Z
\end{equation*}

Fully depolarizing channel (sends everything to maximally mixed state) corresponds to $p = 0.75$.


In [ ]:
dev = qml.device("default.mixed", wires=1)

def prepare_state():
    qml.RY(2 * np.pi/3, wires=0)

@qml.qnode(dev)
def circuit_depolarizing(p):
    prepare_state()
    qml.DepolarizingChannel(p, wires=0)
    return qml.probs()

In [ ]:
circuit_depolarizing(0)

In [ ]:
circuit_depolarizing(0.02)

In [ ]:
@qml.qnode(dev)
def circuit_depolarizing(p):
    prepare_state()
    qml.DepolarizingChannel(p, wires=0)
    return qml.state()

In [ ]:
qml.math.fidelity(circuit_depolarizing(0), circuit_depolarizing(0.05))

In [ ]:
p_values = np.linspace(0., 0.75, 50)

plt.scatter(
    p_values,
    [qml.math.fidelity(circuit_depolarizing(0), circuit_depolarizing(p)) for p in p_values]
)
plt.scatter(
    p_values,
    [qml.math.trace_distance(circuit_depolarizing(0), circuit_depolarizing(p)) for p in p_values]
)
plt.xlabel("Depolarizing strength")
plt.ylabel("Fidelity with original state")

In [ ]:
qml.math.fidelity(circuit_depolarizing(0), np.eye(2)/2)

## Demo 2: Amplitude damping channel

In [ ]:
@qml.qnode(dev)
def circuit_ampdamp(p):
    prepare_state()
    qml.AmplitudeDamping(p, wires=0)
    return qml.probs()

In [ ]:
p_values = np.linspace(0., 1, 50)

plt.scatter(
    p_values,
    [circuit_ampdamp(p)[1] for p in p_values]
)
plt.xlabel("Amplitude damping strength")
plt.ylabel("Probability of observing |1>")

## Demo 3: bit flip channel

In [ ]:
@qml.qnode(dev)
def circuit_bitflip(theta, p):
    qml.Hadamard(wires=0)
    qml.RZ(theta, wires=0)
    qml.BitFlip(p, wires=0)
    return qml.expval(qml.PauliX(0))

@qml.qnode(dev)
def circuit_bitflip_state(theta, p):
    qml.Hadamard(wires=0)
    qml.RZ(theta, wires=0)
    qml.BitFlip(p, wires=0)
    return qml.state()

In [ ]:
p = 0.05
thetas = np.linspace(0., np.pi, 100)

plt.scatter(
    thetas,
    [qml.math.fidelity(circuit_bitflip_state(theta, 0), circuit_bitflip_state(theta, p)) for theta in thetas]
)
plt.plot(thetas, 1 - p + p * np.cos(thetas) ** 2,  c="orange"
)
plt.xlabel("Z rotation angle")
plt.ylabel("Fidelity with initial state")